# Case-04：側移單跨剛架 (Sway Single-Bay Frame)

本版直接從 GitHub 抓取最新的 `sd_framework.py` / `samples/model_sway_frame.py`，不在 notebook 裡內嵌原始碼。

## 題目摘要
| 項目 | 內容 |
|---|---|
| 結構 | 門型剛架，A、D兩柱底固定，柱高H相同，B點承受水平集中載重P |
| H | 4.0 m |
| L | 6.0 m |
| P | 12.0 kN |
| Dk (獨立位移自由度) | 3 (θ_B, θ_C, ψ=Δ/H) |

跟 Case-03 唯一差別：多了側移角 ψ=Δ/H 這個自由度、多一條整體剪力平衡方程式(ΣFx=0)。

## 0. 安裝套件 + 抓取最新原始碼

In [ ]:
try:
    import anastruct
    print("已偵測到 anastruct，略過安裝")
except ImportError:
    print("未偵測到 anastruct，開始安裝...")
    !pip install anastruct -q --break-system-packages
    try:
        import anastruct
        print("安裝完成，可以繼續往下執行")
    except ImportError:
        print("安裝後仍無法載入，這是 Colab 偶爾會發生的快取問題，"
              "請直接重新執行這個 cell 一次（通常第二次就會成功）")

%matplotlib inline

In [ ]:
import sys
for _m in ['sd_framework', 'model_sway_frame']:
    if _m in sys.modules:
        del sys.modules[_m]

!wget -q -O sd_framework.py https://raw.githubusercontent.com/zhixiu0223/slope_deflection_framework/main/sd_framework.py
!wget -q -O model_sway_frame.py https://raw.githubusercontent.com/zhixiu0223/slope_deflection_framework/main/samples/model_sway_frame.py

from sd_framework import SlopeDeflectionSolver, member_shear_curve, member_moment_curve, member_offset_curve
from model_sway_frame import SwayFrameProblem
print("原始碼抓取完成")

## 1. 求解並產生「手寫詳解」風格輸出（步驟1~5）

In [ ]:
problem = SwayFrameProblem(H=4.0, L=6.0, P=12.0, EI_numeric=15000.0)
solver = SlopeDeflectionSolver(problem)
solver.solve_and_report()

## 1b. 教學講義輸出（只有五張圖）

In [ ]:
solver.print_teaching_handout()

## 2. anastruct 獨立建模驗證

In [ ]:
from anastruct import SystemElements
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

H, L, P = 4.0, 6.0, 12.0
ss = SystemElements(EA=1e8, EI=15000.0)
ss.add_element(location=[[0, 0], [0, H]])
ss.add_element(location=[[0, H], [L, H]])
ss.add_element(location=[[L, 0], [L, H]])
ss.add_support_fixed(node_id=1)
ss.add_support_fixed(node_id=4)
ss.point_load(node_id=2, Fx=P)
ss.solve()

print("B點側移:", ss.get_node_results_system(2)['ux'])

def apply_grid(fig, y_lines=()):
    ax = fig.axes[0]
    ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(1))
    ax.grid(True, linestyle='--', alpha=0.4)
    for y in y_lines:
        ax.axhline(y, color='gray', lw=0.6, zorder=0)
    return fig

fig_s = ss.show_structure(show=False, figsize=(7, 6))
apply_grid(fig_s, [0, H])
plt.show()

fig_v = ss.show_shear_force(show=False, figsize=(7, 6))
apply_grid(fig_v, [0, H])
plt.show()

fig_m = ss.show_bending_moment(show=False, figsize=(7, 6))
apply_grid(fig_m, [0, H])
plt.show()

fig_d = ss.show_displacement(show=False, factor=80, figsize=(7, 6))
apply_grid(fig_d, [0, H])
plt.show()

## 3. 反力交叉檢查

In [ ]:
m_ab, m_ba = -14.4, -9.6
m_cd, m_dc = -9.6, -14.4
H_A = (m_ab + m_ba) / H
H_D = (m_dc + m_cd) / H
print(f"我們框架推算: H_A={H_A:.3f}  H_D={H_D:.3f}")
assert abs(H_A + H_D + P) < 0.01
print("\n反力交叉檢查通過 ✓")